# AWS Batch Execution Exploration

This notebook lets you experiment with the new AWS batch execution feature for Datasmith.
You can test both local and AWS execution modes, configure different parameters, and see how the system works.

## Setup
First, let's import the necessary modules and set up some test data.


In [1]:
%cd /home/asehgal/formulacode/datasmith
%load_ext autoreload
%autoreload 2
import logging
import os
from pathlib import Path

import asv

from datasmith.docker.context import ContextRegistry, DockerContext, Task
from datasmith.docker.orchestrator import batch_orchestrate
from datasmith.logging_config import configure_logging
from scratch.notebooks.utils import update_cr

configure_logging(level=logging.INFO)

Couldn't load asv.plugins._mamba_helpers because
No module named 'libmambapy'


/home/asehgal/formulacode/datasmith


09:37:30 WARNING  simple_useragent.core: Falling back to historic user agent.


<Logger datasmith (INFO)>

## Create Test Data

Let's create some test tasks and contexts to work with.


In [2]:
import random
from collections import defaultdict

cr = ContextRegistry.load_from_file(Path("scratch/filtered2_merged_context_registry_2025-09-16T03:26:41.179572.json"))

test_tasks: list[tuple[Task, DockerContext]] = list(update_cr(cr).registry.items())
all_owner_repos = defaultdict(list)
for t, c in test_tasks:
    if "default" in t.repo:
        continue
    all_owner_repos[f"{t.owner}/{t.repo}"].append((t, c))

sampled_tasks = [random.sample(v, 2) if len(v) > 1 else v for v in all_owner_repos.values()]
test_tasks = [(t.with_tag("run"), c) for v in sampled_tasks for (t, c) in v]

print(f"Created {len(test_tasks)} test tasks:")
for task, _ in test_tasks:
    print(task.get_image_name())

# ASV configuration
asv_args = "--append-samples -a rounds=2 -a repeat=2 --python=same"
machine_args: dict[str, str] = asv.machine.Machine.get_defaults()  # pyright: ignore[reportAttributeAccessIssue]
machine_args["num_cpu"] = "4"

print(f"\nASV args: {asv_args}")
print(f"Machine args: {machine_args}")

Created 99 test tasks:
curent-andes-cb03c30548211d6102f1f3a53c8cef2b90a6e777:run
curent-andes-183dee17f7f65a8eef472e23bc6c1ff65a7b4ae3:run
gaa-uam-scikit-fda-d03223cdceea392d2007836ec5e78582d15a38ca:run
gaa-uam-scikit-fda-6781c70983a32d2b326c9f1e8d6a9c0bf8c674e4:run
ncar-geocat-comp-fffe34e43f893f255e5027eb0245aec454f324bc:run
ncar-geocat-comp-85f05087b4cc479d6285d859f429155763b52db8:run
pywavelets-pywt-d95a01fa722244eb16da488a16ea927d725e23b4:run
pywavelets-pywt-3f8fc437d03cd4bf7cfb347c29ac22b32058156e:run
quansight-labs-ndindex-2fb49e97dd7f1b5a036b56f59b4a1665477e6bc6:run
quansight-labs-ndindex-85a15884019e2fb2113563989e90ff7c5470af2a:run
rockhopper-technologies-enlighten-7985d68e971cc843c97e1e94aaa1f802efece43d:run
rockhopper-technologies-enlighten-a4ef0cf7db588d1c6c117f5a428e5892c8d92f72:run
scitools-cartopy-396043e1ff09061a946b6b62c4af482f5ab9f8bb:run
scitools-cartopy-499ded4b17585ca9dd688facb3a10393088f19d5:run
textualize-rich-ef0f5ca24b191633f796b31fc958244d26eb1259:run
textuali

## Test 1: Local Execution (Fallback)

First, let's test the local execution mode to make sure the integration works.


In [3]:
# async def test_local_execution():
#     """Test local execution mode."""
#     print("🧪 Testing local execution mode...")
#     # Create output directory
#     output_dir = Path("datasmith_test_local")
#     output_dir.mkdir(exist_ok=True)

#     try:
#         # For local execution, we need a real Docker client
#         from datasmith.docker.orchestrator import get_docker_client
#         client = get_docker_client(max_concurrency=2)

#         results = await batch_orchestrate(
#             contexts=test_tasks,
#             asv_args=asv_args,
#             machine_args=machine_args,
#             max_concurrency=32,
#             n_cores=2,
#             output_dir=output_dir,
#             client=client,
#             use_aws_batch=False,  # Use local execution
#             aws_batch_config=None
#         )
#         for task, files in results.items():
#             print(f"   - {task.owner}/{task.repo}: {len(files)} result files")

#         return results

#     except Exception as e:
#         print(f"❌ Local execution failed: {e}")
#         print(f"   Error type: {type(e).__name__}")
#         return None

# # Run the test
# local_results = await test_local_execution()


In [4]:
# to_remove = [
# "activitysim",
# "aicsimageio",
# "asdf",
# "chempy",
# "calebbell",
# "dasdae",
# "datalad",
# "devito",
# "dottxt-ai",
# "freegs",
# "datashader",
# "loopy",
# "intelpython",
# "jdasoftwaregroup",
# "janim",
# "oggm",
# "innobi",
# "newton-physics",
# "modin-project",
# "makepath",
# "mars-project",
# "qcodes",
# "sourmash",
# "anndata",
# "contrib-metric-learn",
# "pynetdicom",
# "climpred",
# "nilearn",
# "kedro",
# "mujoco",
# "mongodb-labs",
# "mdanalysis",
# "pvlib",
# "psygnal",
# "nvidia-warp",
# "man-group-arcticdb",
# "pydata-bottleneck",
# "pybamm-team",
# "pydicom-pydicom",
# "pybop-team",
# "python-control",
# "hyper-h11",
# "pymc-devs",
# "pysal-momepy",
# "qiskit",
# "quantumlib-cirq",
# "betterproto",
# "components",
# "django-components",
# "apache-arrow",
# "bloomberg-memray",
# "deepchecks-deepchecks",
# "ipython-ipyparallel",
# "lmfit-lmfit",
# "man-group-arctic",
# "neurostuff",
# "scverse-spatialdata",
# "tensorwerk-hangar",
# "dask",
# "django-components",
# "bloomberg-memray",
# "unidata-metpy",
# "deepchecks-deepchecks",
# "lmfit-lmfit",
# "ipython-ipyparallel",
# "man-group-arctic",
# "scitools-iris",
# "posthog-posthog",
# "scverse-scanpy",
# "stac-utils-pystac",
# "royerlab-ultrack",
# "tensorwerk-hangar",
# "scverse-spatialdata",
# ]
# from copy import deepcopy
# updated_cr = deepcopy(cr)
# print(f"Before filtering, {len(updated_cr.registry)} tasks")
# updated_cr.registry = {k : v for k, v in cr.registry.items() if not any(remove_str in k.get_image_name() for remove_str in to_remove)}
# print(f"After filtering, {len(updated_cr.registry)} tasks")
# updated_cr.save_to_file(Path("scratch/filtered2_merged_context_registry_2025-09-16T03:26:41.179572.json"))

## Test 2: AWS Configuration Validation

Let's test the AWS configuration validation without actually running on AWS.


In [ ]:
async def test_aws_config_validation():
    """Test AWS configuration validation."""
    print("🧪 Testing AWS configuration validation...")
    print("\n2. Testing with complete config (should reach AWS API)...")
    aws_config = {
        "region": os.environ["AWS_REGION"],
        "s3_bucket": os.environ["AWS_S3_BUCKET"],
        "subnet_id": os.environ["AWS_SUBNET_ID"],
        "security_group_ids": [os.environ["AWS_SECURITY_GROUP_ID"]],
        "iam_instance_profile_name": os.environ["AWS_IAM_INSTANCE_PROFILE_NAME"],
        "ami_id": os.environ["AWS_AMI_ID"],
        "instance_type": os.environ["AWS_INSTANCE_TYPE"],
        "max_tasks_per_instance": 20,
        "batch_timeout_s": 5 * 60 * 60,
        "poll_interval_s": 30,
        "max_batch_retries": 1,
        "spot_max_price": "0.17",
    }

    try:
        results = await batch_orchestrate(
            contexts=test_tasks,
            asv_args=asv_args,
            machine_args=machine_args,
            max_concurrency=2,
            n_cores=2,
            output_dir=Path("aws_logs/"),
            client=None,
            use_aws_batch=True,
            aws_batch_config=aws_config,
        )
        return results  # noqa: TRY300
    except Exception as e:
        error_msg = str(e).lower()
        if any(keyword in error_msg for keyword in ["aws", "s3", "ec2", "access", "denied"]):
            print("   ✅ Correctly reached AWS API (failed as expected without real setup)")
            print(f"   Error: {type(e).__name__}")
        else:
            print(f"   ❌ Unexpected error: {e}")
        return None


# Run the test
ouptut2 = await test_aws_config_validation()

09:37:34 INFO     botocore.credentials: Found credentials in shared credentials file: ~/.aws/credentials


🧪 Testing AWS configuration validation...

2. Testing with complete config (should reach AWS API)...


09:37:34 INFO     datasmith.datasmith.docker.aws_batch_executor: Starting batch execution for 99 tasks with run_id=ba3853e9e5404e52b34897e81dc578ba
09:37:34 INFO     datasmith.datasmith.docker.aws_batch_executor: Split 99 tasks into 5 batches
09:37:34 INFO     datasmith.datasmith.docker.aws_batch_executor: Uploaded batch data to s3://test-datasmith-bucket/datasmith-batch-execution/batches/ba3853e9e5404e52b34897e81dc578ba/batch-000/batch-data.json
09:37:34 INFO     datasmith.datasmith.docker.aws_batch_executor: Uploaded batch data to s3://test-datasmith-bucket/datasmith-batch-execution/batches/ba3853e9e5404e52b34897e81dc578ba/batch-001/batch-data.json
09:37:34 INFO     datasmith.datasmith.docker.aws_batch_executor: Uploaded batch data to s3://test-datasmith-bucket/datasmith-batch-execution/batches/ba3853e9e5404e52b34897e81dc578ba/batch-002/batch-data.json
09:37:34 INFO     datasmith.datasmith.docker.aws_batch_executor: Uploaded batch data to s3://test-datasmith-bucket/datasmith-batch-ex

In [ ]:
ouptut2

NameError: name 'ouptut2' is not defined

## Performance Comparison

Let's calculate the theoretical performance improvements for different scenarios.


In [ ]:
def calculate_performance_improvements():
    """Calculate performance improvements for different scenarios."""
    print("⚡ Performance Comparison Calculator")
    print("=" * 50)

    # Local execution assumptions
    local_cores = 8
    local_time_per_task = 0.5  # hours per task (example)

    # AWS execution assumptions
    aws_instances = 50
    aws_cores_per_instance = 4
    aws_time_per_task = 0.3  # hours per task (faster due to dedicated resources)

    scenarios = [
        {"name": "Small Scale", "tasks": 100},
        {"name": "Medium Scale", "tasks": 1000},
        {"name": "Large Scale", "tasks": 5000},
        {"name": "Very Large Scale", "tasks": 10000},
    ]

    print(f"Local Machine: {local_cores} cores, {local_time_per_task}h per task")
    print(f"AWS Batch: {aws_instances} instances x {aws_cores_per_instance} cores, {aws_time_per_task}h per task")
    print()

    print(f"{'Scenario':<15} | {'Tasks':<8} | {'Local Time':<12} | {'AWS Time':<10} | {'Speedup':<8} | {'Cost Est.'}")
    print("-" * 80)

    for scenario in scenarios:
        tasks = scenario["tasks"]

        # Local execution time (sequential with limited cores)
        local_time = (tasks * local_time_per_task) / local_cores

        # AWS execution time (parallel across instances)
        tasks_per_instance = tasks / aws_instances
        aws_time = tasks_per_instance * aws_time_per_task

        # Speedup calculation
        speedup = local_time / aws_time if aws_time > 0 else float("inf")

        # Cost estimation (rough)
        aws_cost = aws_instances * aws_time * 0.10  # $0.10/hour per instance (spot pricing)

        print(
            f"{scenario['name']:<15} | {tasks:<8} | {local_time:>8.1f}h | {aws_time:>6.1f}h | {speedup:>6.1f}x | ${aws_cost:>6.1f}"
        )

    print("\n💡 Key Insights:")
    print("  • AWS batch execution scales linearly with the number of instances")
    print("  • Local execution is limited by your machine's cores")
    print("  • Cost scales with execution time, not task count")
    print("  • Spot instances can reduce costs by up to 90%")


calculate_performance_improvements()

## Interactive Configuration Builder

Build your own AWS configuration interactively.


In [ ]:
def build_aws_config():
    """Interactive AWS configuration builder."""
    print("🔧 Interactive AWS Configuration Builder")
    print("=" * 50)

    # You can modify these values to experiment with different configurations
    config = {
        "region": "us-west-2",  # Change to your preferred region
        "s3_bucket": "your-datasmith-bucket",  # Replace with your bucket
        "subnet_id": "subnet-12345678",  # Replace with your subnet
        "security_group_ids": ["sg-12345678"],  # Replace with your security groups
        "iam_instance_profile_name": "datasmith-batch-execution-role",  # Replace with your role
        "ami_id": "ami-0c02fb55956c7d316",  # Amazon Linux 2023
        "instance_type": "c6i.xlarge",  # Try: c6i.large, c6i.xlarge, c6i.2xlarge
        "key_name": None,  # Optional: SSH key for debugging
        "spot_max_price": "0.50",  # Optional: max price for spot instances
        "tags": {"project": "datasmith", "role": "batch-execution", "environment": "testing"},
        "max_tasks_per_instance": 100,  # Try: 10, 50, 100, 200
        "batch_timeout_s": 2 * 60 * 60,  # 2 hours
        "poll_interval_s": 30,  # Poll S3 every 30 seconds
        "max_batch_retries": 1,  # Retry failed batches once
    }

    print("Current configuration:")
    for key, value in config.items():
        print(f"  {key}: {value}")

    print("\n💡 To modify this configuration:")
    print("  1. Edit the values in the config dictionary above")
    print("  2. Re-run this cell to see your changes")
    print("  3. Use the config in the test functions below")

    return config


# Build the configuration
my_aws_config = build_aws_config()

## Test Your Configuration

Use the configuration you built above to test AWS batch execution.


In [ ]:
async def test_my_configuration():
    """Test with your custom configuration."""
    print("🧪 Testing with your custom configuration...")

    # Use the configuration from the previous cell
    config = my_aws_config

    # Create output directory
    output_dir = Path("datasmith_test_custom")
    output_dir.mkdir(exist_ok=True)

    try:
        results = await batch_orchestrate(
            contexts=test_tasks,
            asv_args=asv_args,
            machine_args=machine_args,
            max_concurrency=1,  # Only 1 batch for testing
            n_cores=4,
            output_dir=output_dir,
            client=None,
            use_aws_batch=True,
            aws_batch_config=config,
        )

        print("✅ Custom configuration test completed!")
        print(f"   Results: {len(results)} tasks processed")
        for task, files in results.items():
            print(f"   - {task.owner}/{task.repo}: {len(files)} result files")

        return results  # noqa: TRY300

    except Exception as e:
        print(f"❌ Custom configuration test failed: {e}")
        print(f"   Error type: {type(e).__name__}")

        # Provide helpful error messages
        error_msg = str(e).lower()
        if "access denied" in error_msg:
            print("   💡 This usually means the S3 bucket doesn't exist or you don't have permissions")
        elif "subnet" in error_msg:
            print("   💡 Check that your subnet ID is correct and in the right region")
        elif "security group" in error_msg:
            print("   💡 Check that your security group IDs are correct")
        elif "iam" in error_msg:
            print("   💡 Check that your IAM instance profile name is correct")

        return None


# Uncomment to test your configuration:
# custom_results = await test_my_configuration()

print("💡 To test your custom configuration, uncomment the line above.")

## Summary and Next Steps

This notebook has demonstrated:

1. ✅ **Local execution fallback** - The system works with your existing local setup
2. ✅ **AWS configuration validation** - Proper error handling for missing/invalid configs
3. ✅ **Performance analysis** - Understanding the speedup and cost benefits
4. ✅ **Configuration exploration** - Different options and their trade-offs

### Next Steps:

1. **Set up AWS infrastructure** if you want to use AWS batch execution:
   - Create an S3 bucket
   - Set up IAM roles with S3 permissions
   - Configure VPC subnet and security groups
   - Get an appropriate AMI ID

2. **Test with real AWS resources** by uncommenting the real AWS test section

3. **Integrate with your existing workflow** by adding `--use-aws-batch` to your command line

4. **Monitor and optimize** based on your actual workload characteristics

### Command Line Usage:

```bash
# Your existing command
python scratch/scripts/benchmark_commits.py --commits commits.jsonl --context-registry registry.json

# With AWS batch execution
python scratch/scripts/benchmark_commits.py \
    --commits commits.jsonl \
    --context-registry registry.json \
    --use-aws-batch \
    --aws-region us-west-2 \
    --aws-s3-bucket your-bucket \
    --aws-subnet-id subnet-12345678 \
    --aws-sg-id sg-12345678 \
    --aws-iam-instance-profile your-role \
    --aws-ami-id ami-0c02fb55956c7d316
```

Happy benchmarking! 🚀
